In [ ]:
"""
海洋人才三维能力聚类分析
基于 valid_three_layer_archive/extraction_results/new_data_prompt_test_10/*.json，对三层
（外层 · 蓝色胜任表征维度 / 中层 · 协同创新支撑维度 / 内层 · 深蓝精神内核维度）
分别进行 BERTopic 聚类和肘部法 K 值评估。

说明：当前 DATA_DIR 指向新 prompt 全量重跑后的 209 个 JSON。
"""

import json
from pathlib import Path
from typing import List

from umap import UMAP
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.cluster import BisectingKMeans, KMeans
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import pandas as pd


# ============================================================
# 0. 全局配置
# ============================================================

# 兼容两种启动位置：
# 1) 在项目总目录启动 notebook；2) 直接在 valid_three_layer_archive/ 内启动 notebook。
PROJECT_ROOT = Path("valid_three_layer_archive") if Path("valid_three_layer_archive").exists() else Path(".")

# 新 prompt 全量重跑结果。目录名沿用试跑时的名称，但现在已包含 209 个 JSON。
DATA_DIR = PROJECT_ROOT / "extraction_results" / "new_data_prompt_test_10"

# 聚类输出仍写回原 topic_results 结构，后续 backfill_topics.py 可继续复用。
TOPIC_ROOT = PROJECT_ROOT / "topic_results"
OUTER_OUT = TOPIC_ROOT / "outer"
MIDDLE_OUT = TOPIC_ROOT / "middle"
INNER_OUT = TOPIC_ROOT / "inner"
TOPIC_NAMES_FILE = TOPIC_ROOT / "topic_names_suggested.csv"

EXPECTED_JSON_COUNT = 209
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# 各层 K 值（先沿用旧版本；跑完肘部图后可再调整）
OUTER_K = 12
MIDDLE_K = 10
INNER_K = 7


# ============================================================
# 1. 数据加载 —— 适配新 prompt 三层 JSON 格式
# ============================================================

def load_layer_signals(data_dir: Path, layer_key: str) -> List[str]:
    """从新 prompt 三层 JSON 加载指定层的所有 signals。"""
    docs = []
    if not data_dir.exists():
        raise FileNotFoundError(f"数据目录不存在: {data_dir.resolve()}")

    json_files = sorted(data_dir.glob("*.json"))
    # 排除临时测试文件；当前正式人物文件名不会包含 _test。
    json_files = [f for f in json_files if "_test" not in f.stem]

    print(f"扫描 {len(json_files)} 个文件 -> 提取 {layer_key} ...")
    if len(json_files) != EXPECTED_JSON_COUNT:
        print(f"  提醒：期望 {EXPECTED_JSON_COUNT} 个 JSON，当前为 {len(json_files)} 个。")

    bad_files = []
    blank_files = []
    for fp in json_files:
        try:
            data = json.loads(fp.read_text(encoding="utf-8"))
        except Exception as exc:
            bad_files.append(f"{fp.name}: JSON读取失败 {exc}")
            continue

        if data.get("_error") or data.get("_parse_error"):
            bad_files.append(f"{fp.name}: API/解析错误占位")
            continue

        layer = data.get(layer_key, {})
        signals = layer.get("signals", []) if isinstance(layer, dict) else []
        cleaned = []
        for s in signals:
            if not isinstance(s, str):
                continue
            s = s.strip()
            if s and len(s) > 1:
                cleaned.append(s)
                docs.append(s)
        if not cleaned:
            blank_files.append(fp.name)

    if bad_files:
        raise RuntimeError("发现错误 JSON，请先重跑这些文件：\n" + "\n".join(bad_files[:20]))

    if blank_files:
        print(f"  {layer_key} 空信号文件 {len(blank_files)} 个：{', '.join(blank_files[:10])}")

    print(f"  {layer_key}: 汇总 signals {len(docs)} 条")
    return docs


def dedup_keep_order(docs: List[str]) -> List[str]:
    seen = set()
    return [d for d in docs if not (d in seen or seen.add(d))]


def archive_stale_topic_names() -> None:
    """重新聚类前归档旧命名表，避免旧 topic 名称误回标到新聚类。"""
    if not TOPIC_NAMES_FILE.exists():
        return
    backup = TOPIC_NAMES_FILE.with_name("topic_names_suggested.before_new_prompt.csv")
    if backup.exists():
        TOPIC_NAMES_FILE.unlink()
        print(f"  已删除旧命名表: {TOPIC_NAMES_FILE}")
    else:
        TOPIC_NAMES_FILE.replace(backup)
        print(f"  已归档旧命名表: {backup}")


# ============================================================
# 2. BERTopic 聚类
# ============================================================

def run_bertopic(
    layer_key: str,
    layer_label: str,
    output_dir: Path,
    n_clusters: int,
    n_neighbors: int = 20,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    docs = load_layer_signals(DATA_DIR, layer_key)
    if not docs:
        print(f"  {layer_label} 无数据，跳过。")
        return
    print(f"  {layer_label}: 原始 {len(docs)} -> 去重后 ", end="")
    docs = dedup_keep_order(docs)
    print(f"{len(docs)}")

    print("  加载 Embedding 模型 ...")
    emb_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    embeddings = emb_model.encode(docs, show_progress_bar=True)

    umap_model = UMAP(
        n_neighbors=n_neighbors,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42,
    )

    vectorizer = CountVectorizer(analyzer=lambda x: [x], lowercase=False)

    print(f"  训练 BisectingKMeans (K={n_clusters}) ...")
    cluster_model = BisectingKMeans(
        n_clusters=n_clusters,
        init="k-means++",
        n_init=10,
        bisecting_strategy="largest_cluster",
        random_state=42,
    )

    topic_model = BERTopic(
        embedding_model=emb_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer,
        language="multilingual",
        calculate_probabilities=False,
        nr_topics=None,
    )
    topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)

    freq = topic_model.get_topic_info()
    print(f"\n  {layer_label} 聚类结果 (前 10):")
    print(freq.head(10))

    freq["CustomLabel"] = freq["Representation"].apply(
        lambda x: " | ".join(x[:3])
    )
    freq.to_csv(str(output_dir / f"{layer_key}_topics.csv"), index=False, encoding="utf-8-sig")
    pd.DataFrame({"Doc": docs, "Topic": topics}).to_csv(
        str(output_dir / f"{layer_key}_assignments.csv"), index=False, encoding="utf-8-sig"
    )

    print("  生成可视化 ...")
    try:
        topic_model.visualize_topics().write_html(
            str(output_dir / f"{layer_key}_visualization.html")
        )
        print(f"  结果已保存至 {output_dir}")
    except Exception as e:
        print(f"  可视化失败: {e}")


# ============================================================
# 3. 肘部法 —— 评估最优 K 值
# ============================================================

def run_elbow(
    layer_key: str,
    layer_label: str,
    output_dir: Path,
    k_range: range = range(2, 21),
    n_neighbors: int = 20,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    docs = load_layer_signals(DATA_DIR, layer_key)
    if not docs:
        print(f"  {layer_label} 无数据，跳过。")
        return
    print(f"  {layer_label}: 原始 {len(docs)} -> 去重后 ", end="")
    docs = dedup_keep_order(docs)
    print(f"{len(docs)}")

    if len(docs) < max(k_range):
        print("  数据量不足，跳过。")
        return

    emb_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    embeddings = emb_model.encode(docs, show_progress_bar=True)

    umap_model = UMAP(
        n_neighbors=n_neighbors,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42,
    )
    umap_embeddings = umap_model.fit_transform(embeddings)

    inertias = []
    print(f"  评估 K in [{k_range.start}, {k_range.stop - 1}] ...")
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
        kmeans.fit(umap_embeddings)
        inertias.append(kmeans.inertia_)

    plt.figure(figsize=(12, 7))
    plt.plot(k_range, inertias, "bo-", markersize=8, linewidth=2)
    plt.title(f"Elbow Method - {layer_label}", fontsize=16)
    plt.xlabel("Number of Clusters (K)", fontsize=14)
    plt.ylabel("Inertia (WCSS)", fontsize=14)
    plt.xticks(k_range)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()

    save_path = output_dir / f"{layer_key}_elbow.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"  肘部图已保存至 {save_path}")
    plt.show()


# ============================================================
# 4. 一键运行某层分析（肘部法 + BERTopic）
# ============================================================

def analyze_layer(
    layer_key: str,
    layer_label: str,
    output_dir: Path,
    n_clusters: int,
    k_range: range = range(2, 21),
    n_neighbors: int = 20,
) -> None:
    print(f"\n{'='*60}")
    print(f"  {layer_label} ({layer_key})")
    print(f"{'='*60}")

    archive_stale_topic_names()

    print("\n--- 肘部法 ---")
    run_elbow(layer_key, layer_label, output_dir, k_range, n_neighbors)

    print(f"\n--- BERTopic 聚类 (K={n_clusters}) ---")
    run_bertopic(layer_key, layer_label, output_dir, n_clusters, n_neighbors)


In [ ]:
# Cell 2: 外层 · 蓝色胜任表征维度 (outer_layer) 肘部法 + BERTopic
analyze_layer("outer_layer", "外层 · 蓝色胜任表征维度", OUTER_OUT, n_clusters=OUTER_K)


In [ ]:
# Cell 3: 中层 · 协同创新支撑维度 (middle_layer) 肘部法 + BERTopic
analyze_layer("middle_layer", "中层 · 协同创新支撑维度", MIDDLE_OUT, n_clusters=MIDDLE_K)


In [ ]:
# Cell 4: 内层 · 深蓝精神内核维度 (inner_layer) 肘部法 + BERTopic
analyze_layer("inner_layer", "内层 · 深蓝精神内核维度", INNER_OUT, n_clusters=INNER_K)


In [ ]:
# Cell 5: 单独跑某一层 BERTopic（按需取消注释）
# run_bertopic("outer_layer", "外层 · 蓝色胜任表征维度", OUTER_OUT, n_clusters=OUTER_K)
# run_bertopic("middle_layer", "中层 · 协同创新支撑维度", MIDDLE_OUT, n_clusters=MIDDLE_K)
# run_bertopic("inner_layer", "内层 · 深蓝精神内核维度", INNER_OUT, n_clusters=INNER_K)


In [ ]:
# Cell 6: 单独跑某一层的肘部法（按需取消注释）
# run_elbow("outer_layer", "外层 · 蓝色胜任表征维度", OUTER_OUT)
# run_elbow("middle_layer", "中层 · 协同创新支撑维度", MIDDLE_OUT)
# run_elbow("inner_layer", "内层 · 深蓝精神内核维度", INNER_OUT)
